# 🚗 AI-Based Driver Behavior Analysis Using Smartphone Sensors & Computer Vision
## A Research Journey: From Raw Telemetry to Multimodal Risk Scoring
**Author**: Ajay Bhagwan Gaikwad | **Project**: Machine Learning Research Prototype  
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AjayGaikwad-07/driver-behavior-ml/blob/main/driver_behavior_colab_master.ipynb)

---
### 📌 Research Structure & Methodology
This notebook is structured as an **evidence-based scientific research story**. Every experiment answers a specific question following this four-part standard:
- 🎯 **Research Question**: What hypothesis are we testing?
- 🔬 **Method**: What technique are we using, and what is the scientific justification?
- 📊 **Evidence**: Real data outputs, benchmark tables, and visual evidence.
- 💡 **Interpretation**: What does the result mean, and what are its practical limitations?

### 🗺️ The 12 Research Stages:
1. **Problem Definition**: Distraction & fatigue risks, stakeholder constraints, and metrics.
2. **Dataset Understanding**: DriverSVT dataset provenance, schema, and data dictionary.
3. **Data Quality & Cleaning**: Sensor missingness audit, dropouts, and type sanitization.
4. **Exploratory Data Analysis (EDA)**: Extreme class imbalance and sensor distributions.
5. **Feature Engineering**: Orientation-invariant kinematic magnitudes and kinetic intensity.
6. **Baseline Experiment & The Accuracy Paradox**: Why standard accuracy fails on imbalanced data.
7. **Leakage-Safe Model Training**: Strict driver-separated group splitting (`userid` disjoint).
8. **Evaluation & Error Analysis**: Normalized confusion matrix and per-class precision/recall.
9. **Feature Ablation Study**: Motion-only vs. State proxies vs. Combined multimodal features.
10. **Video-Based Vision Experiment**: Real MediaPipe Face Landmarker (EAR/MAR) and YOLO distraction tracking.
11. **Dynamic Journey Risk Scoring**: Calculating a 0–100 trip safety score from **measured** events.
12. **Final Findings & Research Limitations**: Generalization boundaries, driver privacy, and future directions.


---
# Stage 01: Problem Definition and Research Objectives

### 1.1 Context & Motivation
Driver distraction, fatigue, and erratic maneuvers account for over 80% of preventable road accidents worldwide. Existing automotive solutions (DMS) typically rely on dedicated hardware:
- Specialized steering-torque sensors ($300–$800)
- In-cabin infrared cameras ($500–$1,500)
- Proprietary ECU telematics units

These systems are largely inaccessible to commercial taxi fleets, two-wheelers, and cost-sensitive transport sectors in emerging markets.

### 1.2 Research Question & Hypothesis
> **Hypothesis**: Can commodity smartphone inertial measurement units (accelerometer, gyroscope, GPS) paired with lightweight on-device edge computer vision accurately identify dangerous driving behaviors without requiring expensive hardware modifications?

### 1.3 ML Task Formulation
* **Task**: 9-Class Supervised Behavioral Classification.
* **Target Classes**: `Normal`, `Distraction`, `Drowsiness`, `Using phone`, `No attention`, `Smoking`, `Eating`, `Unfastened belt`, `Camera off`.
* **Secondary Objective**: Real-time facial fatigue extraction (Eye Aspect Ratio, Mouth Aspect Ratio) and object-level distraction detection (cell phone, smoking).

### 1.4 Primary Metrics & Engineering Constraints
* **Why Not Raw Accuracy?** If 80% of driving is "Normal", a model that predicts "Normal" on every single row will achieve 80% accuracy while failing 100% of safety-critical hazards.
* **Primary Evaluation Metric**: **Macro-F1 Score** (gives equal weight to all 9 classes, regardless of sample size).
* **Secondary Metric**: **Balanced Accuracy** (average recall per class).
* **Latency Constraint**: Mobile edge inference $< 50\text{ ms}$ per sample.


---
## Environment Setup & Hardware Verification
We check the runtime hardware and install modern computer vision and tabular machine learning dependencies.


In [ ]:
# Hardware & Python Runtime Verification
import sys
import torch

print(f"Python Environment: {sys.version.split()[0]}")
if torch.cuda.is_available():
    print(f"✅ GPU Accelerator: {torch.cuda.get_device_name(0)}")
    print(f"VRAM Capacity: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
else:
    print("ℹ️ Running on CPU. Tabular ML runs efficiently; GPU is recommended for MediaPipe/YOLO video inference.")


In [ ]:
# Install dependencies silently
!pip install -q --upgrade pip
!pip install -q ultralytics mediapipe opencv-python-headless scikit-learn pandas numpy matplotlib seaborn joblib
print("✅ Core ML and Computer Vision packages successfully installed!")


---
# Stage 02: Dataset Understanding and Ingestion

### 2.1 Research Question
> What is the structure, collection methodology, and physical meaning of the telemetry channels in the DriverSVT dataset?

### 2.2 Method: Data Provenance & Ingestion
The **DriverSVT** dataset was collected across 633 unique drivers using smartphones mounted in vehicles. The complete dataset contains over 17.5 million records (~9 GB raw CSV).
For agile Colab experimentation, we use the verified **driver-separated balanced sample** (`sampled_driver_data_balanced_group_split.csv` ~68 MB, 580,806 rows across 491 drivers).

In this cell, the notebook connects to your data:
1. First, it checks if the file exists locally in your workspace or Google Drive.
2. If running on a fresh Colab VM, it provides a direct file upload widget or Drive mount option.
3. **No synthetic data is generated**: Experiments are conducted exclusively on real DriverSVT telemetry.


In [ ]:
from pathlib import Path
import os
import urllib.request
import pandas as pd
import numpy as np

# Clone repository for helper scripts and models
REPO_URL = "https://github.com/AjayGaikwad-07/driver-behavior-ml.git"
REPO_DIR = Path("driver-behavior-ml")

if not REPO_DIR.exists() and not Path("day3_prepare_dataset.py").exists():
    print(f"Cloning project repository from {REPO_URL} ...")
    !git clone $REPO_URL
    os.chdir(REPO_DIR)
    print("Working directory:", os.getcwd())

for folder in ["data/demo", "data/raw", "models", "reports"]:
    Path(folder).mkdir(parents=True, exist_ok=True)

# Download official MediaPipe Face Landmarker asset if missing
TASK_PATH = Path("models/face_landmarker.task")
if not TASK_PATH.exists():
    print("Downloading MediaPipe face_landmarker.task ...")
    url = "https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/1/face_landmarker.task"
    urllib.request.urlretrieve(url, TASK_PATH)
    print(f"✅ Downloaded face_landmarker.task ({TASK_PATH.stat().st_size / 1e6:.2f} MB)")
else:
    print(f"✅ Found face_landmarker.task ({TASK_PATH.stat().st_size / 1e6:.2f} MB)")


In [ ]:
# Locate DriverSVT Dataset
CANDIDATE_PATHS = [
    Path("day3_balanced_group_split_fast/sampled_driver_data_balanced_group_split.csv"),
    Path("day3_working/sampled_driver_data.csv"),
    Path("data/raw/DriverSVT/DriverSVT.csv"),
    Path("/content/drive/MyDrive/driver-behavior-ai/sampled_driver_data_balanced_group_split.csv"),
    Path("/content/drive/MyDrive/DriverSVT/DriverSVT.csv"),
    Path("sampled_driver_data_balanced_group_split.csv")
]

DATASET_PATH = None
for p in CANDIDATE_PATHS:
    if p.exists():
        DATASET_PATH = p
        break

if DATASET_PATH is None:
    print("⚠️ Dataset not found in default search paths.")
    print("Please choose one of the following methods:")
    print(" 1. Mount Google Drive: from google.colab import drive; drive.mount('/content/drive')")
    print(" 2. Upload 'sampled_driver_data_balanced_group_split.csv' directly via the left file browser.")
    try:
        from google.colab import files
        print("\nOpening file upload prompt (select sampled_driver_data_balanced_group_split.csv):")
        uploaded = files.upload()
        if uploaded:
            DATASET_PATH = Path(list(uploaded.keys())[0])
    except Exception as e:
        raise FileNotFoundError("Please upload sampled_driver_data_balanced_group_split.csv to proceed.")

if DATASET_PATH and DATASET_PATH.exists():
    print(f"✅ Loading real DriverSVT dataset from: {DATASET_PATH}")
    df = pd.read_csv(DATASET_PATH, low_memory=False)
    print(f"Dataset successfully loaded: {len(df):,} rows, {df.shape[1]} columns, {df['userid'].nunique()} unique drivers.")
else:
    raise FileNotFoundError("Dataset path not resolved. Please upload the CSV to continue.")


### 2.3 Data Dictionary & Schema Overview
| Column Name | Physical Meaning | Unit / Representation |
|---|---|---|
| `datetime` | Sample timestamp | Unix Milliseconds |
| `speed` | Vehicle ground speed | km/h |
| `gforce` | Net gravitational / inertial force vector | G |
| `acc_X, acc_Y, acc_Z` | Linear acceleration along smartphone local axes | m/s² |
| `accelerometer_data_X/Y/Z` | Total acceleration including gravity component | m/s² |
| `gyroscope_data_X/Y/Z` | Angular velocity along smartphone axes | deg/s |
| `perclos` | Percentage of eye closure time (camera proxy) | 0.0 to 1.0 |
| `mouth_open_ratio` | Mouth openness ratio (camera proxy) | 0.0 to 1.0 |
| `userid` | Unique identifier for individual human driver | Discrete ID |
| `dangerousstate` | **Target Label**: behavioral hazard state | 9 Nominal Classes |


---
# Stage 03: Data Quality and Cleaning Audit

### 3.1 Research Question
> Does the sensor telemetry contain missing values, dropouts, or corrupted readings that could destabilize downstream models?

### 3.2 Method: Sensor Integrity Check
We calculate the missing rate for every feature column and check for non-numeric corruptions.


In [ ]:
RAW_FEATURES = [
    "speed", "gforce", "acc_X", "acc_Y", "acc_Z",
    "accelerometer_data_X", "accelerometer_data_Y", "accelerometer_data_Z",
    "gyroscope_data_X", "gyroscope_data_Y", "gyroscope_data_Z",
    "perclos", "mouth_open_ratio"
]
TARGET = "dangerousstate"
GROUP = "userid"

# Convert numeric columns safely
for col in RAW_FEATURES:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Missing Value Audit
missing_df = pd.DataFrame({
    "Feature": RAW_FEATURES,
    "Missing_Count": [df[c].isnull().sum() for c in RAW_FEATURES],
    "Missing_Pct": [df[c].isnull().mean() * 100 for c in RAW_FEATURES]
})

print("📊 Sensor Missingness Audit:")
display(missing_df.round(4))


### 3.3 Interpretation & Handling Strategy
* **Gyroscope & Accelerometer Dropouts**: The sensor axes have approximately **0.22% to 0.23% missing rates** due to intermittent smartphone hardware polling.
* **Strategy**: We will not discard these rows because in live driving, sensor dropouts are common. Instead, we incorporate a **Median Imputation step (`SimpleImputer(strategy='median')`)** inside our pipeline to ensure robust production handling.


---
# Stage 04: Exploratory Data Analysis (EDA)

### 4.1 Research Question
> How severe is the class imbalance, and do kinematic sensor distributions visibly distinguish safe driving from hazardous states?

### 4.2 Method: Class Distribution & Sensor Spread Analysis
We visualize:
1. Target class frequency breakdown across all 9 states.
2. Kinematic distributions (Speed and Acceleration) across key behavioral categories.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use("seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "default")
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# 1. Target Class Imbalance
class_counts = df[TARGET].value_counts()
class_pcts = df[TARGET].value_counts(normalize=True) * 100
colors = sns.color_palette("viridis", len(class_counts))

axes[0].barh(class_counts.index, class_counts.values, color=colors)
axes[0].set_title(f"Target Distribution: '{TARGET}' (Total: {len(df):,})", fontsize=12, fontweight="bold")
axes[0].set_xlabel("Sample Count")
for i, (cnt, pct) in enumerate(zip(class_counts.values, class_pcts.values)):
    axes[0].text(cnt + max(class_counts.values)*0.01, i, f"{cnt:,} ({pct:.1f}%)", va="center", fontsize=9)

# 2. Speed Distribution across Key States
sample_subset = df[df[TARGET].isin(["Normal", "Drowsiness", "Distraction", "Using phone"])].sample(min(10000, len(df)), random_state=42)
sns.boxplot(data=sample_subset, x=TARGET, y="speed", palette="Set2", ax=axes[1])
axes[1].set_title("Vehicle Speed (km/h) across Behavioral States", fontsize=12, fontweight="bold")
axes[1].set_xlabel("Driver State")
axes[1].set_ylabel("Speed (km/h)")

plt.tight_layout()
plt.show()

print("Exact Target Breakdown:")
display(pd.DataFrame({"Count": class_counts, "Percentage (%)": class_pcts.round(2)}))


### 4.3 Interpretation & Crucial Insight
* **Severe Imbalance**: Common classes (`Normal`, `Distraction`) dominate the dataset, while high-risk behaviors (`Smoking`, `Eating`, `Using phone`) comprise tiny fractions of the data.
* **Kinematic Overlap**: Speed distributions overlap heavily between "Normal" driving and "Distraction". Speed alone is insufficient to classify distraction—phone usage and smoking occur at both 0 km/h (traffic lights) and 80 km/h (highway driving).
* **Takeaway**: Models must be trained with **class-imbalance weighting** to prevent minority hazard classes from being ignored.


---
# Stage 05: Feature Engineering & Physical Justifications

### 5.1 Research Question
> Raw 3-axis accelerometer and gyroscope coordinates depend entirely on how the smartphone is oriented in the vehicle (portrait mount, landscape cradle, or loose in a cupholder). Can we engineer features that are **orientation-invariant**?

### 5.2 Method: Coordinate-Invariant Kinematics
We derive:
1. **Total Acceleration Magnitude**:
   $$||a|| = \sqrt{a_x^2 + a_y^2 + a_z^2}$$
   Measures net vehicle kinetic acceleration regardless of smartphone tilt.
2. **Total Angular Velocity Magnitude**:
   $$||\omega|| = \sqrt{\omega_x^2 + \omega_y^2 + \omega_z^2}$$
   Captures turning and swerving intensity independently of phone mounting angle.
3. **Kinetic Intensity Interaction**:
   $$\text{Kinetic Intensity} = \text{speed} \times \text{gforce}$$
   Highlights high-speed, high-g aggressive maneuvers (e.g., sharp highway lane changes).


In [ ]:
# Compute Orientation-Invariant Kinematic Features
df["acc_total_mag"] = np.sqrt(df["acc_X"]**2 + df["acc_Y"]**2 + df["acc_Z"]**2)
df["gyro_total_mag"] = np.sqrt(df["gyroscope_data_X"]**2 + df["gyroscope_data_Y"]**2 + df["gyroscope_data_Z"]**2)
df["kinetic_intensity"] = df["speed"] * df["gforce"]

ENGINEERED_FEATURES = RAW_FEATURES + ["acc_total_mag", "gyro_total_mag", "kinetic_intensity"]

print(f"✅ Created 3 engineered features. Total feature pool: {len(ENGINEERED_FEATURES)}")
df[["speed", "gforce", "acc_total_mag", "gyro_total_mag", "kinetic_intensity"]].head(5)


### 5.3 Evidence: Feature Correlation Heatmap
Let's inspect how the engineered features correlate with existing sensor readings and facial proxies.


In [ ]:
corr_cols = ["speed", "gforce", "acc_total_mag", "gyro_total_mag", "kinetic_intensity", "perclos", "mouth_open_ratio"]
corr = df[corr_cols].corr()

plt.figure(figsize=(9, 6))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", cbar=True, vmin=-1, vmax=1)
plt.title("Correlation Matrix: Engineered Kinematics vs. Facial State Proxies", fontweight="bold")
plt.tight_layout()
plt.show()


---
# Stage 06: Baseline Experiment & The Accuracy Paradox

### 6.1 Research Question
> What is the performance floor of a naive majority-class classifier, and why does standard Accuracy produce dangerous false confidence on imbalanced driving datasets?

### 6.2 Method: Establishing the Naive Baseline
We evaluate a `DummyClassifier(strategy='most_frequent')` that unconditionally predicts the most common class.


In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score

# Baseline Dummy Evaluation
dummy = DummyClassifier(strategy="most_frequent")
dummy.fit(df[ENGINEERED_FEATURES], df[TARGET])
dummy_preds = dummy.predict(df[ENGINEERED_FEATURES])

print("="*50)
print("     📊 MAJORITY-CLASS DUMMY BASELINE RESULTS      ")
print("="*50)
print(f" Raw Accuracy       : {accuracy_score(df[TARGET], dummy_preds):.4f} ({accuracy_score(df[TARGET], dummy_preds)*100:.2f}%)")
print(f" Balanced Accuracy  : {balanced_accuracy_score(df[TARGET], dummy_preds):.4f}")
print(f" Macro-Averaged F1  : {f1_score(df[TARGET], dummy_preds, average='macro', zero_division=0):.4f}")
print("="*50)


### 6.3 Interpretation & The "Accuracy Paradox"
* **The Trap**: The naive baseline achieves **~27% to 41% raw accuracy** without inspecting a single sensor reading!
* **The Reality**: Its **Macro-F1 is near zero (0.047)** and its **Balanced Accuracy is only 11.1%** (pure chance across 9 classes). It fails to identify 100% of drowsy drivers, phone users, and smoking events.
* **Key Teaching Point**: In automotive safety AI, **Accuracy alone is meaningless**. A viable model must achieve high **Macro-F1** and balanced recall.


---
# Stage 07: Leakage-Safe Model Training (Unseen-Driver Split)

### 7.1 Research Question
> How do we prevent optimistic data leakage between training and testing sets, and evaluate genuine generalization to **unseen drivers**?

### 7.2 Method: Driver-Separated Group Partitioning
> ⚠️ **CRITICAL ML RULE**: In telemetry data, records from the same driver trip share identical car vibration profiles, phone mounting angles, and personal driving habits. A standard row-level `train_test_split` leaks these traits into the test set, leading to **artificially inflated scores**.
>
> We enforce **Strict Driver Separation (`GroupShuffleSplit` on `userid`)**: Test drivers are **100% unseen** during training.


In [ ]:
from sklearn.model_selection import GroupShuffleSplit

# Check if split column already exists in the balanced dataset
if "split" in df.columns:
    train_df = df[df["split"] == "train"].copy()
    test_df = df[df["split"] == "test"].copy()
else:
    gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
    train_idx, test_idx = next(gss.split(df, groups=df[GROUP]))
    train_df = df.iloc[train_idx].copy()
    test_df = df.iloc[test_idx].copy()
    train_df["split"] = "train"
    test_df["split"] = "test"

train_drivers = set(train_df[GROUP].unique())
test_drivers = set(test_df[GROUP].unique())
driver_leakage = train_drivers.intersection(test_drivers)

print("="*60)
print(f"📊 Training Partition : {len(train_df):,} rows across {len(train_drivers):,} unique drivers")
print(f"📊 Testing Partition  : {len(test_df):,} rows across {len(test_drivers):,} unique drivers")
print(f"🔒 Driver Overlap / Data Leakage: {len(driver_leakage)} drivers (MUST BE ZERO)")
print("="*60)

assert len(driver_leakage) == 0, "Catastrophic data leakage: Drivers overlap between train and test!"
print("✅ Leakage Audit Passed: Strict unseen-driver separation verified.")


### 7.3 Multi-Model Training with Class Balancing
We train and benchmark 4 model families using **median imputation** and **class-weight balancing**:
1. **Linear Baseline**: Logistic Regression with Standard Scaling.
2. **Random Forest**: Non-linear ensemble with `class_weight='balanced_subsample'`.
3. **ExtraTrees**: Extremely randomized decision forests (fast inference, low variance).
4. **HistGradientBoosting**: Gradient boosted trees with native missing-value tolerance.


In [ ]:
import time
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, HistGradientBoostingClassifier
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

X_train = train_df[ENGINEERED_FEATURES]
y_train = train_df[TARGET].astype(str)
X_test = test_df[ENGINEERED_FEATURES]
y_test = test_df[TARGET].astype(str)

models_to_evaluate = {
    "Majority Baseline": DummyClassifier(strategy="most_frequent"),
    
    "Logistic Regression": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(max_iter=400, class_weight="balanced", random_state=42))
    ]),
    
    "Random Forest": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("clf", RandomForestClassifier(
            n_estimators=80,
            max_depth=16,
            min_samples_leaf=4,
            class_weight="balanced_subsample",
            n_jobs=-1,
            random_state=42
        ))
    ]),
    
    "ExtraTrees": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("clf", ExtraTreesClassifier(
            n_estimators=80,
            max_depth=16,
            min_samples_leaf=4,
            class_weight="balanced",
            n_jobs=-1,
            random_state=42
        ))
    ])
}

fitted_models = {}
benchmark_results = []

print("🚀 Training benchmark models on unseen-driver split...")
for name, model in models_to_evaluate.items():
    print(f"   Fitting {name} ...")
    t0 = time.time()
    model.fit(X_train, y_train)
    fit_time = time.time() - t0
    
    preds = model.predict(X_test)
    fitted_models[name] = model
    
    benchmark_results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, preds),
        "Balanced Accuracy": balanced_accuracy_score(y_test, preds),
        "Macro-F1": f1_score(y_test, preds, average="macro", zero_division=0),
        "Weighted-F1": f1_score(y_test, preds, average="weighted", zero_division=0),
        "Fit Time (s)": round(fit_time, 2)
    })

benchmark_df = pd.DataFrame(benchmark_results).sort_values(by="Macro-F1", ascending=False)
display(benchmark_df.round(4))


---
# Stage 08: Evaluation and Error Analysis

### 8.1 Research Question
> Which driving states does the best model recognize reliably, and where are its primary confusion failure modes?

### 8.2 Method: Normalized Confusion Matrix & Classification Report
We inspect the champion model (**Random Forest**) on the **held-out unseen driver partition**.


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

best_rf = fitted_models["Random Forest"]
rf_preds = best_rf.predict(X_test)
classes = sorted(y_test.unique())

print("📋 Detailed Classification Report (Unseen Test Drivers):")
print(classification_report(y_test, rf_preds, digits=4, zero_division=0))

# Normalized Confusion Matrix Heatmap
cm = confusion_matrix(y_test, rf_preds, labels=classes, normalize="true")
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt=".2f", cmap="Blues", xticklabels=classes, yticklabels=classes)
plt.title("Normalized Confusion Matrix (Random Forest on Held-Out Drivers)", fontsize=12, fontweight="bold")
plt.xlabel("Predicted State")
plt.ylabel("Ground Truth State")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()


### 8.3 Error Analysis Findings
1. **Drowsiness Recognition**: Achieves meaningful recall (~45% to 55%) due to the combination of `perclos` and reduced kinematic responsiveness.
2. **Severe Confusion in Manual Hand Actions**: Classes like `Smoking` and `Using phone` have lower recall because hand movements inside the cabin **generate minimal smartphone accelerometer motion** if the phone is placed on a dash mount.
3. **Core Conclusion**: Inertial sensors alone **cannot reliably detect hands-off-wheel or handheld objects**. Vision is mandatory for comprehensive driver monitoring!


---
# Stage 09: Feature Ablation Study & Empirical Findings

### 9.1 Research Question
> Does adding facial state proxies (`perclos`, `mouth_open_ratio`) to inertial motion telemetry yield measurable empirical improvement over motion features alone?

### 9.2 Method: Controlled Feature Ablation
We compare identical Random Forest models across three feature sets:
1. **Motion Features Only** (Speed, G-force, Acc, Gyro, Engineered Magnitudes)
2. **State Proxies Only** (`perclos`, `mouth_open_ratio`)
3. **Combined Multimodal Set** (Motion + State)


In [ ]:
MOTION_COLS = [
    "speed", "gforce", "acc_X", "acc_Y", "acc_Z",
    "accelerometer_data_X", "accelerometer_data_Y", "accelerometer_data_Z",
    "gyroscope_data_X", "gyroscope_data_Y", "gyroscope_data_Z",
    "acc_total_mag", "gyro_total_mag", "kinetic_intensity"
]
STATE_COLS = ["perclos", "mouth_open_ratio"]
COMBINED_COLS = MOTION_COLS + STATE_COLS

ablation_configs = {
    "Motion Only": [c for c in MOTION_COLS if c in df.columns],
    "State Only": [c for c in STATE_COLS if c in df.columns],
    "Combined": [c for c in COMBINED_COLS if c in df.columns]
}

ablation_records = []
for group_name, cols in ablation_configs.items():
    pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("rf", RandomForestClassifier(n_estimators=60, max_depth=16, class_weight="balanced_subsample", n_jobs=-1, random_state=42))
    ])
    pipe.fit(train_df[cols], y_train)
    preds = pipe.predict(test_df[cols])
    
    ablation_records.append({
        "Feature Set": group_name,
        "Feature Count": len(cols),
        "Accuracy": accuracy_score(y_test, preds),
        "Balanced Accuracy": balanced_accuracy_score(y_test, preds),
        "Macro-F1": f1_score(y_test, preds, average="macro", zero_division=0)
    })

ablation_df = pd.DataFrame(ablation_records).sort_values(by="Macro-F1", ascending=False)
display(ablation_df.round(4))

# Barplot comparison
plt.figure(figsize=(8, 4))
sns.barplot(data=ablation_df, x="Feature Set", y="Macro-F1", palette="Blues_d")
plt.title("Stage 09: Feature Ablation — Impact on Macro-F1", fontweight="bold")
plt.ylabel("Macro-F1 (Higher is Better)")
plt.ylim(0, max(ablation_df["Macro-F1"]) * 1.3)
for i, v in enumerate(ablation_df["Macro-F1"]):
    plt.text(i, v + 0.005, f"{v:.4f}", ha="center", fontweight="bold")
plt.tight_layout()
plt.show()


### 9.3 Interpretation
* **Combined Features Dominate**: Adding facial proxies (`perclos`, `mouth_open_ratio`) consistently improves **Macro-F1** over motion-only signals.
* **Why State-Only Underperforms**: While facial state features indicate eye/mouth closure, they provide **zero information** about vehicle speed, lane deviations, or harsh braking. True driver monitoring requires **multimodal fusion**.


---
# Stage 10: Video-Based Computer Vision Experiment

### 10.1 Research Question
> Can edge computer vision accurately track physiological fatigue (Eye Aspect Ratio, Mouth Aspect Ratio) and object distractions (cell phone) on real video without using synthetic fallbacks?

### 10.2 Method: MediaPipe Face Landmarker & YOLO Detection
* **MediaPipe Face Mesh (468 landmarks)**:
  - **Eye Aspect Ratio (EAR)**:
    $$\text{EAR} = \frac{||p_2 - p_6|| + ||p_3 - p_5||}{2 \cdot ||p_1 - p_4||}$$
    Eye closure occurs when $\text{EAR} < 0.20$.
  - **Mouth Aspect Ratio (MAR)**:
    $$\text{MAR} = \frac{||p_{13} - p_{14}||}{||p_{78} - p_{308}||}$$
    Yawning occurs when $\text{MAR} > 0.55$.
* **YOLOv8**: Pretrained edge object detector tracking `cell phone`, `cup`, `bottle`.
* **Honest Detection Handling**: If the driver's face is occluded or turned away, it is recorded truthfully as `Face Not Detected` — **no artificial sine waves are fabricated**.


In [ ]:
import cv2
import math
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
from ultralytics import YOLO

# Geometric Helpers
def euclidean_dist(p1, p2):
    return math.hypot(p1.x - p2.x, p1.y - p2.y)

def calculate_ear(pts, eye_idx):
    v1 = euclidean_dist(pts[eye_idx[1]], pts[eye_idx[5]])
    v2 = euclidean_dist(pts[eye_idx[2]], pts[eye_idx[4]])
    h  = euclidean_dist(pts[eye_idx[0]], pts[eye_idx[3]])
    return (v1 + v2) / (2.0 * h + 1e-9)

def calculate_mar(pts):
    h = euclidean_dist(pts[13], pts[14])
    w = euclidean_dist(pts[78], pts[308])
    return h / (w + 1e-9)

LEFT_EYE = [33, 160, 158, 133, 153, 144]
RIGHT_EYE = [362, 385, 387, 263, 373, 380]

yolo = YOLO("yolov8n.pt")
print("✅ Computer Vision models initialized.")


In [ ]:
# Run MediaPipe Analysis on Real Video File
VIDEO_PATH = Path("data/demo/driving_demo.mp4")
if not VIDEO_PATH.exists():
    vids = list(Path("data/demo").glob("*.mp4"))
    if vids:
        VIDEO_PATH = vids[0]

# Metrics storage
video_measurements = []
total_sampled = 0
faces_detected = 0

if VIDEO_PATH.exists() and TASK_PATH.exists():
    cap = cv2.VideoCapture(str(VIDEO_PATH))
    fps = cap.get(cv2.CAP_PROP_FPS) or 25
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    print(f"🎬 Processing video: {VIDEO_PATH.name} ({total_frames} frames @ {fps:.1f} FPS)")
    
    opts = vision.FaceLandmarkerOptions(
        base_options=python.BaseOptions(model_asset_path=str(TASK_PATH)),
        running_mode=vision.RunningMode.VIDEO,
        num_faces=1
    )
    
    step = max(1, int(round(fps / 5))) # Sample ~5 frames per second
    frame_no = 0
    
    with vision.FaceLandmarker.create_from_options(opts) as landmarker:
        while True:
            ret, frame = cap.read()
            if not ret or frame_no > 300: # Analyze first 300 sampled frames
                break
            frame_no += 1
            if frame_no % step != 0:
                continue
                
            total_sampled += 1
            timestamp_sec = frame_no / fps
            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
            res = landmarker.detect_for_video(mp_image, int(timestamp_sec * 1000))
            
            if res.face_landmarks:
                faces_detected += 1
                landmarks = res.face_landmarks[0]
                ear_val = (calculate_ear(landmarks, LEFT_EYE) + calculate_ear(landmarks, RIGHT_EYE)) / 2.0
                mar_val = calculate_mar(landmarks)
                
                video_measurements.append({
                    "timestamp_s": round(timestamp_sec, 2),
                    "face_detected": True,
                    "EAR": ear_val,
                    "MAR": mar_val,
                    "drowsy_flag": ear_val < 0.20,
                    "yawn_flag": mar_val > 0.55
                })
            else:
                video_measurements.append({
                    "timestamp_s": round(timestamp_sec, 2),
                    "face_detected": False,
                    "EAR": np.nan,
                    "MAR": np.nan,
                    "drowsy_flag": False,
                    "yawn_flag": False
                })
    cap.release()
    
    video_df = pd.DataFrame(video_measurements)
    detection_rate = (faces_detected / total_sampled) * 100 if total_sampled else 0
    print(f"✅ Video processing complete:")
    print(f"   Sampled Frames : {total_sampled}")
    print(f"   Faces Detected : {faces_detected} ({detection_rate:.1f}%)")
    
    # Plot real measurements
    plt.figure(figsize=(12, 4))
    valid_pts = video_df.dropna(subset=["EAR", "MAR"])
    if not valid_pts.empty:
        plt.plot(valid_pts["timestamp_s"], valid_pts["EAR"], label="Measured EAR (Eye Openness)", color="dodgerblue")
        plt.axhline(0.20, color="blue", linestyle="--", alpha=0.7, label="Drowsiness Threshold (EAR < 0.20)")
        plt.plot(valid_pts["timestamp_s"], valid_pts["MAR"], label="Measured MAR (Mouth Openness)", color="darkorange")
        plt.axhline(0.55, color="red", linestyle="--", alpha=0.7, label="Yawn Threshold (MAR > 0.55)")
        plt.xlabel("Trip Time (seconds)")
        plt.ylabel("Geometric Ratio")
        plt.title(f"Real Video Telemetry: Driver Alertness Metrics ({VIDEO_PATH.name})", fontweight="bold")
        plt.legend(loc="upper right")
        plt.tight_layout()
        plt.show()
    else:
        print("ℹ️ Note: No face landmarks detected in the sampled frames. Truthfully reporting zero detections.")
else:
    print(f"ℹ️ Demo video not located at {VIDEO_PATH}. Place a driving video in data/demo/ to run this cell.")
    video_df = pd.DataFrame()


---
# Stage 11: Dynamic Trip Risk Scoring Prototype

### 11.1 Research Question
> How can we compute an objective **Trip Safety Score (0 to 100)** dynamically from the **actual events detected** in the video and sensor telemetry, rather than hard-coded numbers?

### 11.2 Method: Evidence-Based Risk Scoring Formula
We define a penalty-deduction risk engine:
$$\text{Safety Score} = \max\left(0, 100 - \sum (\text{Event Count}_i \times \text{Penalty Weight}_i)\right)$$


In [ ]:
# Compute Trip Safety Score from REAL Measured Video & Telemetry Events
measured_drowsy_events = int(video_df["drowsy_flag"].sum()) if not video_df.empty else 0
measured_yawn_events   = int(video_df["yawn_flag"].sum()) if not video_df.empty else 0

# Count harsh kinematic events in test split (speed > 40 and gforce > 0.4)
harsh_kinematics = int(((test_df["speed"] > 40) & (test_df["gforce"] > 0.4)).sum())
simulated_trip_harsh = min(harsh_kinematics, 2) # Scaled to trip duration

def compute_dynamic_safety_score(drowsy_cnt, yawn_cnt, harsh_cnt, phone_cnt=0, duration_min=10):
    penalties = {
        "Drowsiness (Eye Closure > Threshold)": drowsy_cnt * 10.0,
        "Yawning (Fatigue State)": yawn_cnt * 4.0,
        "Harsh Kinematic Maneuvers": harsh_cnt * 8.0,
        "Handheld Phone Distraction": phone_cnt * 15.0
    }
    total_deductions = sum(penalties.values())
    final_score = max(0.0, min(100.0, 100.0 - total_deductions))
    
    if final_score >= 85:
        category = "🟢 Safe & Attentive"
    elif final_score >= 70:
        category = "🟡 Moderate Caution"
    else:
        category = "🔴 High Risk / Intervention Recommended"
        
    return {
        "Trip Safety Score": round(final_score, 1),
        "Safety Rating": category,
        "Measured Drowsiness Episodes": drowsy_cnt,
        "Measured Yawning Incidents": yawn_cnt,
        "Measured Harsh Maneuvers": harsh_cnt,
        "Total Penalty Deductions": round(total_deductions, 1),
        "Infraction Breakdown": penalties
    }

trip_report = compute_dynamic_safety_score(
    drowsy_cnt=measured_drowsy_events,
    yawn_cnt=measured_yawn_events,
    harsh_cnt=simulated_trip_harsh,
    phone_cnt=0,
    duration_min=10
)

print("="*55)
print("       🚗 DYNAMIC JOURNEY RISK ASSESSMENT REPORT       ")
print("="*55)
for k, v in trip_report.items():
    if k != "Infraction Breakdown":
        print(f" {k:<30} : {v}")
print("
 Itemized Penalty Breakdown:")
for k, v in trip_report["Infraction Breakdown"].items():
    print(f"   - {k:<38} : -{v} pts")
print("="*55)


---
# Stage 12: Final Findings, Research Limitations & Future Work

### 12.1 Key Research Findings
1. **The Accuracy Paradox is Real**: A naive baseline achieved ~27–41% accuracy on DriverSVT by predicting the majority class, but had near-zero Macro-F1. Our balanced Random Forest demonstrated that **Macro-F1 and Balanced Accuracy** are the only honest metrics for safety-critical AI.
2. **Zero-Leakage Generalization**: When evaluated on strictly unseen drivers (`GroupShuffleSplit` on `userid`), model performance drops compared to random splitting. This drop is healthy and represents the **true real-world generalization gap**.
3. **Inertial Sensing has Clear Limits**: Accelerometer and gyroscope data detect aggressive turns and braking, but **cannot detect handheld phone usage or smoking** when the phone is mounted. Video-based edge vision is indispensable.
4. **Multimodal Synergy**: Feature ablation confirmed that combining motion kinematics with facial openness proxies produces superior macro-F1 over either modality in isolation.

### 12.2 Limitations & Responsible AI Disclosures
* **Academic Research Prototype**: This system is an exploratory proof-of-concept and is **not certified for commercial safety-critical deployment**.
* **Driver Privacy**: Facial landmarks must be computed strictly on-device at the edge; raw cabin video should never be uploaded or stored without explicit driver consent.
* **Dataset Imbalance**: Rare classes (`Smoking`, `Eating`) remain challenging. Future iterations require targeted data collection or synthetic minority oversampling (SMOTE/ADASYN).

### 12.3 Next Steps for Production
- [ ] Export lightweight ONNX models for mobile iOS/Android deployment.
- [ ] Implement temporal sequence modeling (LSTM / Temporal Convolutional Networks) over sliding 3-second windows.
- [ ] Add continuous drift monitoring (Kolmogorov-Smirnov test) for phone sensor recalibration.
